In [0]:
# ----------------------------------------
# 1️⃣ Imports
# ----------------------------------------
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

# ----------------------------------------
# 2️⃣ Load trained model (full absolute path)
# ----------------------------------------
MODEL_PATH = "/Workspace/Repos/win185@ensign.edu/Databricks/etl/final_random_forest_model.pkl"
model = joblib.load(MODEL_PATH)

print("Model expects:", model.n_features_in_, "features")

# ----------------------------------------
# 3️⃣ Load sampled dataset (your 20% sample)
# ----------------------------------------
df = spark.table("workspace.silver.labeled_step_test_sampled").toPandas()

print("Sample size:", len(df))

# ----------------------------------------
# 4️⃣ Select the exact 3 features model was trained on
# ----------------------------------------
feature_cols = ["distance_cm", "test_time", "total_steps"]

X = df[feature_cols]

# Recreate label exactly as done during training
y = (df["total_steps"] > 0).astype(int)

# ----------------------------------------
# 5️⃣ Recreate same train/test split
# IMPORTANT: random_state must match training
# ----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42   # <-- change ONLY if your training used a different value
)

print("X_test shape:", X_test.shape)

# ----------------------------------------
# 6️⃣ Generate predictions
# ----------------------------------------
y_pred = model.predict(X_test)

# ----------------------------------------
# 7️⃣ Create predictions DataFrame
# ----------------------------------------
predictions_df = pd.DataFrame({
    "actual_label": y_test,
    "predicted_label": y_pred
})

spark_df = spark.createDataFrame(predictions_df)

# ----------------------------------------
# 8️⃣ Create schema safely (if not exists)
# ----------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.analytics")

# ----------------------------------------
# 9️⃣ Save predictions table safely (NO overwrite)
# ----------------------------------------
spark_df.write.mode("error") \
    .saveAsTable("workspace.analytics.stedi_predictions_capstone")

print("✅ Predictions table created safely.")